In [5]:
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch
import os
import json

from transformers import OwlViTProcessor, OwlViTForObjectDetection

import matplotlib.pyplot as plt
import matplotlib.patches as patches

processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

base_dir = os.getcwd()
coco_root = os.path.join(base_dir, "val2017")
ann_file = os.path.join(base_dir, "annotations", "instances_val2017.json")

coco = COCO(ann_file)
cat_ids = coco.getCatIds()
cat_names = [coco.loadCats(i)[0]["name"] for i in cat_ids]
text_queries = [[f"a photo of a {name}" for name in cat_names]]

loading annotations into memory...
Done (t=0.22s)
creating index...
index created!


In [6]:
detections = []
img_ids = coco.getImgIds()

for img_id in img_ids:
    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(coco_root, img_info["file_name"])
    image = Image.open(img_path).convert("RGB")

    inputs = processor(text=text_queries, images=image, return_tensors="pt")
    outputs = model(**inputs)

    target_sizes = torch.tensor([(image.height, image.width)])
    result = processor.post_process_grounded_object_detection(
        outputs=outputs,
        target_sizes=target_sizes,
        threshold=0.1,
        text_labels=text_queries,
    )[0]

    boxes, scores, labels = result["boxes"], result["scores"], result["text_labels"]

    for box, score, label in zip(boxes, scores, labels):
        x_min, y_min, x_max, y_max = box.tolist()
        w, h = x_max - x_min, y_max - y_min

        try:
            cat_name = label.split("a photo of a ")[-1]
            cat_id = next(
                cid for cid in cat_ids if coco.loadCats(cid)[0]["name"] == cat_name
            )
        except StopIteration:
            continue

        detections.append(
            {
                "image_id": img_id,
                "category_id": cat_id,
                "bbox": [x_min, y_min, w, h],
                "score": float(score.item()),
            }
        )

with open("owlvit_coco_results.json", "w") as f:
    json.dump(detections, f)

In [7]:
coco_dt = coco.loadRes("owlvit_coco_results.json")
evaluator = COCOeval(coco, coco_dt, iouType="bbox")
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

Loading and preparing results...
DONE (t=0.14s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=5.17s).
Accumulating evaluation results...
DONE (t=1.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.272
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.428
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.288
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.095
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.280
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.467
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.255
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.370
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.378
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10